img = cv2.imread(path+"Page"+Page+".jpg")- These codes splits the digital archived copies into page units. 

- The codes are executed in the following order.
    1. Remove redundant spaces around a digital copies.
    
        <=Removes the areas of the image covered with while color.
        
    2. Cuts the page at the border of the two pages.
        
        Borders are detected by searching for a black ertical line.

Code 1: Define the functions and install packages

In [1]:
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import cv2
import os
import pandas as pd

def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Clean(img,Loc,Cut):
    hh, ww = img.shape[:2]
    # convert to grayscale
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # threshold gray image
    thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

    # count number of non-zero pixels in each row. 
    countRow = np.count_nonzero(thresh, axis=1)

    ### Find location of biggest jumps/falls.
    if Loc=='Top':
        peaks, _ = find_peaks(countRow, distance=15)
        HeightTop=abs(np.diff(countRow[peaks][(peaks<hh//Cut)]))
        max1Height=np.sort(HeightTop)[-1]
        Top=peaks[ : -1][abs(np.diff(countRow[peaks]))==max1Height][0]
        
        cropped=img[Top:ww, 0:ww] 
        return(cropped)

    if Loc=='Btm':
        peaks, _ = find_peaks(countRow, distance=15)
        HeightBtm=abs(np.diff(countRow[peaks][(peaks>(Cut-1)*hh//Cut)]))
        max2Height=np.sort(HeightBtm)[-1]
        Btm=peaks[ : -1][abs(np.diff(countRow[peaks]))==max2Height][0]

        cropped=img[0:Btm, 0:ww] 
        return(cropped)

def CleanH(img,Dir,Cut):
    hh, ww = img.shape[:2]
    # convert to grayscale
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # threshold gray image
    thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

    # count number of non-zero pixels in each row. 
    countCol = np.count_nonzero(thresh, axis=0)

    ### Find location of biggest jumps/falls.
    if Dir=='Left':
        peaks, _ = find_peaks(countCol, distance=15)
        HeightLeft=abs(np.diff(countCol[peaks][(peaks<ww//Cut)]))
        max1Height=np.sort(HeightLeft)[-1]
        Left=peaks[ : -1][abs(np.diff(countCol[peaks]))==max1Height][0]
        
        cropped=img[0:hh, Left:ww] 
        return(cropped)

    if Dir=='Right':
        peaks, _ = find_peaks(countCol, distance=15)
        HeightRigh=abs(np.diff(countCol[peaks][(peaks>(Cut-1)*ww//Cut)]))
        max2Height=np.sort(HeightRigh)[-1]
        Righ=peaks[ : -1][abs(np.diff(countCol[peaks]))==max2Height][0]

        cropped=img[0:hh, 0:Righ] 
        return(cropped)


def Split(Page,Cut):    
    img = cv2.imread(path+"Page"+Page+".jpg")
    HH, WW = img.shape[:2]
    # convert to grayscale
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # threshold gray image
    thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

    # count number of non-zero pixels in each column and row. 
    countCol = np.count_nonzero(thresh, axis=0)

    ### This finds the height of the lowest valley
    peaks, _ = find_peaks(-countCol, distance=15)
    maxHeight=np.sort(abs(np.diff(countCol[peaks][(peaks<4*WW//7) & (peaks>3*WW//7)])))[-1]
    Border=peaks[ : -1][abs(np.diff(countCol[peaks]))==maxHeight][0]
    Border=625

    #Leftside
    result = img.copy()
    LeftImage=result[0:HH,0:Border]
    RighImage=result[0:HH,Border:WW]
    
    HH,WW=LeftImage.shape[:2]
    
    UpLeft=LeftImage[0:HH//2 -15,0:WW]
    UpLeft=Clean(UpLeft,'Top',Cut)
    UpLeft=CleanH(UpLeft,'Left',Cut)
    
    BtLeft=LeftImage[HH//2 -15:HH,0:WW]
    BtLeft=Clean(BtLeft,'Btm',Cut)
    BtLeft=CleanH(BtLeft,'Left',Cut)
    
    LeftImage=list((UpLeft,BtLeft))
    
    #Rightside
    HH,WW=RighImage.shape[:2]
    UpRigh=RighImage[0:HH//2 -15,0:WW]
    UpRigh=Clean(UpRigh,'Top',Cut)
    UpRigh=CleanH(UpRigh,'Right',Cut)
    
    BtRigh=RighImage[HH//2 -15:HH,0:WW]
    BtRigh=Clean(BtRigh,'Btm',Cut)
    BtRigh=CleanH(BtRigh,'Right',Cut)
    
    RighImage=list((UpRigh,BtRigh))
    #cv2.imshow("LeftImage", LeftImage)
    #cv2.imshow("RightImage", RighImage)
    #cv2.waitKey(0)

    return(list((LeftImage,RighImage)))

In [2]:
def Check(Year):
    for Page in range(StrPage,EndPage):
        file_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+str(Year)+"\\"
        file_path=os.path.join(file_path,"Page"+"{:03d}".format(Page))
        img1=cv2.imread(file_path+'\\Top.jpg')
        img2=cv2.imread(file_path+'\\Btm.jpg')
        
        print('Page:'+str(Page))
        cv2.imshow('Top',img1)
        cv2.waitKey(0)
        cv2.imshow('Btm',img2)
        cv2.waitKey(0)
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [5]:
path=r'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs/Raw_Data\\Metropolitan_DA\\'+str(Year)+'\\Line\\'
x=625
for Page in range(13,300):
    Img=cv2.imread(path+"Page"+"{:03d}".format(Page)+'.jpg')
    cv2.line(Img,(x,0),(x,250),(225,0,0),2)
    cv2.imshow('Sample',Img)
    cv2.waitKey(0)
cv2.destroyAllWindows()
cv2.waitKey(0)

error: OpenCV(4.7.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:971: error: (-215:Assertion failed) size.width>0 && size.height>0 in function 'cv::imshow'


Code 2: Define Years

In [13]:
Year=1940

Code 3: We need to create a seperate directory for each page of the archived copy.
Check the starting and ending page of the staff directory and define a variable with the numbers.

In [6]:
StrPage=11
EndPage=112

#Set directory path
save_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+str(Year)+"\\"

#Create directory for each page
for Num in range(StrPage,EndPage+1):
    Page="Page"+"{:03d}".format(2*(Num-StrPage))
    path2 = os.path.join(save_path, Page)
    if not os.path.exists(path2):
        os.mkdir(path2)
    
    Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
    path2 = os.path.join(save_path, Page)
    if not os.path.exists(path2):
        os.mkdir(path2)

Code 4: Splitting copies into pages.
Each page is saved into the corresponding directory that we made on Code 2.

In [7]:
# read image+crop image
import os
FailedList=[]
path=r'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs/Raw_Data\\Metropolitan_DA\\'+str(Year)+'\\Line\\'
for Num in range(StrPage,EndPage):
    try:
        print(Num)
        Righ=Split("{:03d}".format(Num),5)[1]
        Left=Split("{:03d}".format(Num),5)[0]

        Page="Page"+"{:03d}".format(2*(Num-StrPage))
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Righ[0])
        cv2.imwrite(path2+"\\Btm.jpg", Righ[1])

        Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Left[0])
        cv2.imwrite(path2+"\\Btm.jpg", Left[1])
    except:
        print("Error at Page " +str(Num))
        FailedList.append(Num)
        continue

11
12
13
14
15
16
17
18
Error at Page 18
19
20
Error at Page 20
21
22
23
24
25
26
Error at Page 26
27
Error at Page 27
28
Error at Page 28
29
30
31
32
33
34
35
36
37
Error at Page 37
38
39
Error at Page 39
40
41
42
43
44
45
46
47
48
49
50
51
Error at Page 51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
Error at Page 67
68
69
70
Error at Page 70
71
72
73
Error at Page 73
74
75
76
Error at Page 76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
Error at Page 91
92
93
94
95
96
97
98
Error at Page 98
99
Error at Page 99
100
Error at Page 100
101
Error at Page 101
102
103
Error at Page 103
104
105
106
107
Error at Page 107
108
Error at Page 108
109
110
111
Error at Page 111


In [8]:
FailedList2=[]
for Num in FailedList:
    try:
        print(Num)
        Righ=Split("{:03d}".format(Num),4)[1]
        Left=Split("{:03d}".format(Num),4)[0]

        Page="Page"+"{:03d}".format(2*(Num-StrPage))
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Righ[0])
        cv2.imwrite(path2+"\\Btm.jpg", Righ[1])

        Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Left[0])
        cv2.imwrite(path2+"\\Btm.jpg", Left[1])
    except:
        print("Error at Page " +str(Num))
        FailedList2.append(Num)
        continue

18
20
26
27
28
37
Error at Page 37
39
51
Error at Page 51
67
Error at Page 67
70
Error at Page 70
73
76
91
Error at Page 91
98
99
100
Error at Page 100
101
103
107
Error at Page 107
108
111


In [9]:
FailedList3=[]
for Num in FailedList2:
    try:
        print(Num)
        Righ=Split("{:03d}".format(Num),3)[1]
        Left=Split("{:03d}".format(Num),3)[0]

        Page="Page"+"{:03d}".format(2*(Num-StrPage))
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Righ[0])
        cv2.imwrite(path2+"\\Btm.jpg", Righ[1])

        Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Left[0])
        cv2.imwrite(path2+"\\Btm.jpg", Left[1])
    except:
        print("Error at Page " +str(Num))
        FailedList3.append(Num)
        continue

37
51
67
70
91
100
107


In [11]:
FailRate=len(FailedList3)/(EndPage-StrPage)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(round(1-FailRate,3))+". Fix "+str(len(FailedList3))+" Pages")
else:
    print('残念...Success Rate is '+str(1-FailRate))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index_col=False)
DF.loc[int(Year)-1934, 'Page'] = round(1-FailRate,3)
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

Fantastic!! Success Rate is 1.0. Fix 0 Pages


,Year,Parts,WritePage,Page,PageFin,Office,OfficeFin,Unit,UnitFin,Position,PositionFin,Data
0,1934,1,1,0.985714286,.,0.9,.,NaN,.,0.08,.,.
1,1935,1,1,1,.,0.984375,.,NaN,.,-0.09375,.,.
2,1936,1,0.235294118,.,.,0.235294118,.,NaN,.,.,.,.
3,1937,1,1,1,.,0.865671642,.,NaN,.,.,.,.
4,1938,1,1,1,.,1,.,0.988095238,.,0.98816568,.,0.619957537
5,1939,1,1,1,.,0.927835052,.,1,.,.,.,.
6,1940,2,1,1.0,.,0.991803279,.,NaN,.,.,.,.
7,1941,2,1,1,.,0.565217391,1,NaN,.,.,.,.
8,1942,2,1,1,.,.,.,NaN,.,.,.,.
9,1943,2,1,.,.,.,.,.,.,.,.,.


In [12]:
Check(Year)

Page:11
Page:12
Page:13
Page:14
Page:15
Page:16
Page:17
Page:18
Page:19
Page:20
Page:21
Page:22
Page:23
Page:24
Page:25
Page:26
Page:27
Page:28
Page:29
Page:30
Page:31
Page:32
Page:33
Page:34
Page:35
Page:36
Page:37
Page:38
Page:39
Page:40
Page:41
Page:42
Page:43
Page:44
Page:45
Page:46
Page:47
Page:48
Page:49
Page:50
Page:51
Page:52
Page:53
Page:54
Page:55
Page:56
Page:57
Page:58
Page:59
Page:60
Page:61
Page:62
Page:63
Page:64
Page:65
Page:66
Page:67
Page:68
Page:69
Page:70
Page:71
Page:72
Page:73
Page:74
Page:75
Page:76
Page:77
Page:78
Page:79
Page:80
Page:81
Page:82
Page:83
Page:84
Page:85
Page:86
Page:87
Page:88
Page:89
Page:90
Page:91
Page:92
Page:93
Page:94
Page:95
Page:96
Page:97
Page:98
Page:99
Page:100
Page:101
Page:102
Page:103
Page:104
Page:105
Page:106
Page:107
Page:108
Page:109
Page:110
Page:111


In [21]:
Type1=[[15,'Btm'],
       [50,'Top'],
       [57,'Btm']]

In [16]:
Type1_2=[]
for Num in [d[0] for d in Type1]:
    try:
        print(Num)
        Righ=Split("{:03d}".format(Num),5)[1]
        Left=Split("{:03d}".format(Num),5)[0]

        Page="Page"+"{:03d}".format(2*(Num-StrPage))
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Righ[0])
        cv2.imwrite(path2+"\\Btm.jpg", Righ[1])

        Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\Top.jpg", Left[0])
        cv2.imwrite(path2+"\\Btm.jpg", Left[1])
    except:
        print("Error at Page " +str(Num))
        Type1_2.append(Num)
        continue

25
48
60


In [24]:
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'
df = pd.DataFrame(Type1, columns =['Page', 'Part'], dtype = int)
save_path=os.path.join(save_path,str(Year),"Page")
if not os.path.exists(save_path):
    os.mkdir(save_path)
df.to_csv(save_path+'\\Type1.csv',index=False)

C:\Temp\ipykernel_8416\1455583618.py:2: FutureWarning: Could not cast to int32, falling back to object. This behavior is deprecated. In a future version, when a dtype is passed to 'DataFrame', either all columns will be cast to that dtype, or a TypeError will be raised.
  df = pd.DataFrame(Type1, columns =['Page', 'Part'], dtype = int)


In [19]:
Check(Year)

Page:12
Page:13
Page:14
Page:15
Page:16
Page:17
Page:18
Page:19
Page:20
Page:21
Page:22
Page:23
Page:24
Page:25
Page:26
Page:27
Page:28
Page:29
Page:30
Page:31
Page:32
Page:33
Page:34
Page:35
Page:36
Page:37
Page:38
Page:39
Page:40
Page:41
Page:42
Page:43
Page:44
Page:45
Page:46
Page:47
Page:48
Page:49
Page:50
Page:51
Page:52
Page:53
Page:54
Page:55
Page:56
Page:57
Page:58
Page:59
Page:60
Page:61
Page:62
Page:63
Page:64
Page:65
Page:66
Page:67
Page:68
Page:69
Page:70
Page:71
Page:72
Page:73
Page:74
Page:75
Page:76
Page:77
Page:78
Page:79
Page:80
Page:81
Page:82
Page:83
Page:84
Page:85
Page:86
Page:87
Page:88
Page:89
Page:90
Page:91
Page:92
Page:93
Page:94
Page:95
Page:96
Page:97
Page:98
Page:99
Page:100
Page:101
Page:102
Page:103
Page:104
Page:105
Page:106
Page:107
Page:108
Page:109
Page:110
Page:111
